# Compare — Clean vs Dirty (original vs fixed)

Side-by-side diff of the **per-(day,bond) market yield `Y[d,i]`** the backtest trades on, computed
two ways: the **original** folder (clean price) vs **this** folder (dirty price). This is where the
fix bites — `Y` drives the swap trigger and the turnover.

### How to produce the two input files (one-time)
Both backtests already define `export_panels(path=...)` right after the Section-1 precompute (the
`Y[d,i]` / `DUR` / `TAU` cell). You do **not** need to run the full daily backtest or Gurobi for the
panel diff — just run Sections through that precompute, then call `export_panels(...)` with the paths
below.

**In the ORIGINAL `../FABN_Optimizer_SAP_Backtest.ipynb`** (after the precompute cell), add a cell:
```python
export_panels(r"<THIS_FOLDER>/panels_clean.npz")
```
**In THIS folder's `FABN_Optimizer_SAP_Backtest.ipynb`** (after the precompute cell), add a cell:
```python
export_panels(r"<THIS_FOLDER>/panels_dirty.npz")
```
where `<THIS_FOLDER>` is the absolute path printed in the next cell. Then run this notebook.


In [9]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt

HERE = os.getcwd()
CLEAN = os.path.join(HERE, "panels_clean.npz")
DIRTY = os.path.join(HERE, "panels_dirty.npz")
print("THIS_FOLDER =", HERE)
print("expecting:\n ", CLEAN, "\n ", DIRTY)

def load(path):
    z = np.load(path, allow_pickle=True)
    return dict(Y=z["Y"], MID=z["MID"], ELIG=z["ELIG"].astype(bool),
                CUSIPS=list(z["CUSIPS"]),
                dates=pd.to_datetime(z["BT_DATES_ns"].astype("datetime64[ns]")))

have = os.path.exists(CLEAN) and os.path.exists(DIRTY)
if not have:
    print("\n>>> Missing one or both panel files. Produce them with export_panels(...) as in the header,")
    print("    then re-run. (clean exists:", os.path.exists(CLEAN), "| dirty exists:", os.path.exists(DIRTY), ")")
else:
    A = load(CLEAN); B = load(DIRTY)
    assert A["CUSIPS"] == B["CUSIPS"] and len(A["dates"]) == len(B["dates"]), "panels not aligned"
    print("\nloaded:", A["Y"].shape, "days x bonds")

THIS_FOLDER = /Users/eloibernier/Documents/MS-ADS/2026_Spring/Capstone_I/Repo/Pyxidr-Capstone-Project/Optimization/Clean_vs_Dirty_Price_Fix
expecting:
  /Users/eloibernier/Documents/MS-ADS/2026_Spring/Capstone_I/Repo/Pyxidr-Capstone-Project/Optimization/Clean_vs_Dirty_Price_Fix/panels_clean.npz 
  /Users/eloibernier/Documents/MS-ADS/2026_Spring/Capstone_I/Repo/Pyxidr-Capstone-Project/Optimization/Clean_vs_Dirty_Price_Fix/panels_dirty.npz

>>> Missing one or both panel files. Produce them with export_panels(...) as in the header,
    then re-run. (clean exists: False | dirty exists: False )


## 1 — Yield-level difference (clean overstates dirty)
Over every eligible (day, bond), `Y_clean − Y_dirty` should be ≥ 0 (clean overstated) and on the
order of tens of bps.

In [10]:
if have:
    m = A["ELIG"] & B["ELIG"] & np.isfinite(A["Y"]) & np.isfinite(B["Y"]) & (A["Y"]!=0) & (B["Y"]!=0)
    dY = (A["Y"][m] - B["Y"][m]) * 1e4   # bps
    print(f"eligible (day,bond) points : {m.sum():,}")
    print(f"Y_clean - Y_dirty (bps)    : mean {dY.mean():+.1f} | median {np.median(dY):+.1f} "
          f"| p95 {np.percentile(dY,95):+.1f} | max {dY.max():+.1f}")
    print(f"fraction with clean > dirty: {(dY>0).mean()*100:.1f}%  (expect ~100%)")
    fig, ax = plt.subplots(figsize=(8,3.5))
    ax.hist(dY, bins=80, color="#2980b9"); ax.set_title("Y_clean - Y_dirty  (bps, all eligible day×bond)")
    ax.set_xlabel("bps overstatement"); ax.axvline(0, color="k", lw=0.8); plt.tight_layout(); plt.show()
else: print("skipped")

skipped


## 2 — The sawtooth (the real reason turnover is affected)
For a single bond, the **clean** yield drifts up through each coupon period and snaps down at the
coupon (a sawtooth); the **dirty** yield is smooth. That sawtooth is what fires phantom swaps.

In [11]:
if have:
    # pick a high-coupon, long-lived, mostly-eligible bond for a clear sawtooth
    elig_count = (A["ELIG"] & B["ELIG"]).sum(axis=0)
    swing = np.where(elig_count>0, np.nanmax(np.where(A["ELIG"],A["Y"],np.nan),axis=0)
                                  - np.nanmin(np.where(A["ELIG"],A["Y"],np.nan),axis=0), 0)
    i = int(np.nanargmax(np.where(elig_count> A["Y"].shape[0]*0.5, swing, -1)))
    d = A["dates"]; e = A["ELIG"][:,i] & B["ELIG"][:,i]
    fig, ax = plt.subplots(figsize=(10,4))
    ax.plot(d[e], A["Y"][e,i]*100, color="#e67e22", lw=1.2, label="clean (original)")
    ax.plot(d[e], B["Y"][e,i]*100, color="#2980b9", lw=1.6, label="dirty (fixed)")
    ax.set_title(f"Market yield Y[d,i] for bond {A['CUSIPS'][i]} — clean sawtooth vs dirty smooth")
    ax.set_ylabel("yield (%)"); ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    print("If the orange line saws and the blue is smooth, that's the phantom-trade source removed.")
else: print("skipped")

skipped


## 3 — Thesis numbers (fill from the two full backtests)

The panel diff above is the *mechanism*. The *headline* (does the +10–13% thesis and turnover hold)
comes from running each full backtest. Run `FABN_Optimizer_SAP_Backtest.ipynb` end-to-end in **both**
folders and record the Section-4 / Section-1B numbers here:

| Metric | Original (clean) | Fixed (dirty) | Read |
|---|---|---|---|
| Static net ($M) | … | … | level shifts down with dirty |
| Dynamic base net (daily, k=1) ($M) | … | … | |
| Dynamic **tuned** net ($M) | … | … | |
| Tuned advantage vs static (%) | … | … | **thesis: should stay positive** |
| Tuned **turnover** ($M) | … | … | **watch: should drop (sawtooth gone)** |
| Mean in-force book yield (%) | … | … | drops by the measured bias |

**Optional auto-compare:** if you save the result frames in each backtest, e.g.
`sta_df.to_csv("sta_clean.csv"); dyn_tuned_df.to_csv("dyn_clean.csv")` (original) and `..._dirty.csv`
(fixed) into THIS folder, the next cell tabulates them.

In [12]:
import glob
def _net(p): return pd.read_csv(p)["Net"].sum()/1e6 if os.path.exists(p) else None
rows=[]
for tag in ["sta","dyn"]:
    c=os.path.join(HERE,f"{tag}_clean.csv"); d=os.path.join(HERE,f"{tag}_dirty.csv")
    if os.path.exists(c) or os.path.exists(d):
        rows.append(dict(series=tag, clean_M=_net(c), dirty_M=_net(d)))
if rows:
    t=pd.DataFrame(rows); t["delta_M"]=t["dirty_M"]-t["clean_M"]; display(t)
else:
    print("No *_clean.csv / *_dirty.csv found — fill the table above by hand, or export the CSVs.")

No *_clean.csv / *_dirty.csv found — fill the table above by hand, or export the CSVs.
